# 주인을 알아보는 로봇청소기 — TensorFlow 12단계 실습

OpenCV Haar Cascade + 사전학습 MobileNetV2 + 코사인 유사도. CNN 가중치를 새로 학습하지 않으며, OWNER 후보는 검증된 신원 판별이 아닙니다. 개인 얼굴사진·생체 벡터는 공개 GitHub에 업로드하지 마세요.

## STEP 1. 환경 확인

In [ ]:
import tensorflow as tf, numpy as np, cv2, matplotlib.pyplot as plt
print("TensorFlow:",tf.__version__,"NumPy:",np.__version__)
print("GPU:",tf.config.list_physical_devices("GPU"))

## STEP 2. faces.zip 업로드

In [ ]:
from google.colab import files
uploaded=files.upload()
assert "faces.zip" in uploaded
print(list(uploaded))

## STEP 3. 압축 해제와 원본 사진 확인

In [ ]:
import zipfile
from pathlib import Path
with zipfile.ZipFile("faces.zip") as z:
    assert all(not n.startswith("/") and ".." not in Path(n).parts for n in z.namelist())
    z.extractall("data")
owner_dir=Path("data/owner")
if not owner_dir.exists():
    owner_dir=next(p for p in Path("data").rglob("owner") if p.is_dir())
image_files=sorted([*owner_dir.glob("*.jpg"),*owner_dir.glob("*.jpeg"),*owner_dir.glob("*.png")])
assert image_files
print("사진:",len(image_files),[p.name for p in image_files])

## STEP 4. OpenCV 얼굴 검출

In [ ]:
face_detector=cv2.CascadeClassifier(cv2.data.haarcascades+"haarcascade_frontalface_default.xml")
assert not face_detector.empty()
face_images=[]
for path in image_files:
    image=cv2.imread(str(path))
    if image is None: continue
    boxes=face_detector.detectMultiScale(cv2.cvtColor(image,cv2.COLOR_BGR2GRAY),scaleFactor=1.1,minNeighbors=5,minSize=(40,40))
    if len(boxes)==0: print("얼굴 없음:",path); continue
    x,y,w,h=max(boxes,key=lambda b:b[2]*b[3])
    face_images.append(cv2.resize(cv2.cvtColor(image[y:y+h,x:x+w],cv2.COLOR_BGR2RGB),(224,224)))
face_images=np.asarray(face_images)
assert len(face_images)
print(face_images.shape)
plt.figure(figsize=(12,4))
for i,img in enumerate(face_images[:8]):
    plt.subplot(2,4,i+1);plt.imshow(img);plt.axis("off")
plt.tight_layout();plt.show()

## STEP 5. CNN 입력 전처리

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
X=preprocess_input(face_images.astype(np.float32).copy())
print("Shape:",X.shape,"Range:",X.min(),X.max())

## STEP 6. 원본 8장 → 증강 40장

In [ ]:
augmentation=tf.keras.Sequential([tf.keras.layers.RandomRotation(.05),tf.keras.layers.RandomTranslation(.05,.05),tf.keras.layers.RandomZoom(.1),tf.keras.layers.RandomContrast(.1)])
augmented_images=[]
for face in face_images:
    batch=tf.expand_dims(tf.cast(face,tf.float32),0)
    for _ in range(5): augmented_images.append(augmentation(batch,training=True)[0].numpy())
augmented_images=np.asarray(augmented_images,dtype=np.float32)
print("증강:",augmented_images.shape)
plt.figure(figsize=(12,5))
for i in range(min(8,len(augmented_images))):
    plt.subplot(2,4,i+1);plt.imshow(np.clip(augmented_images[i]/255,0,1));plt.axis("off")
plt.tight_layout();plt.show()

## STEP 7. MobileNetV2 사전학습 CNN 불러오기

In [ ]:
base_model=tf.keras.applications.MobileNetV2(input_shape=(224,224,3),include_top=False,weights="imagenet",pooling="avg")
base_model.trainable=False
print("출력:",base_model.output_shape)
base_model.summary()

## STEP 8. 48×1280 CNN 특징 추출

In [ ]:
all_images=np.concatenate([face_images.astype(np.float32),augmented_images.astype(np.float32)])
all_X=preprocess_input(all_images.copy())
features=base_model.predict(all_X,batch_size=16,verbose=1)
print("Feature shape:",features.shape)
print(features[0][:20])

## STEP 9. 주인 대표 벡터 생성

In [ ]:
def normalize(v):
    v=np.asarray(v,dtype=np.float32)
    return v/(np.linalg.norm(v,axis=-1,keepdims=True)+1e-10)
normalized_features=normalize(features)
owner_vector=normalize(normalized_features.mean(axis=0))
owner_similarities=normalized_features[:len(face_images)]@owner_vector
print("Owner vector:",owner_vector.shape,"Norm:",np.linalg.norm(owner_vector))
for i,s in enumerate(owner_similarities):print(f"원본 {i+1}: {s:.4f}")

## STEP 10. 유사도 통계와 시연 기준

In [ ]:
print("Min/Max/Mean/Std:",owner_similarities.min(),owner_similarities.max(),owner_similarities.mean(),owner_similarities.std())
plt.bar(range(1,len(owner_similarities)+1),owner_similarities)
plt.axhline(owner_similarities.mean(),color="red",linestyle="--")
plt.xlabel("Owner Image Number");plt.ylabel("Cosine Similarity");plt.ylim(0,1.05);plt.show()
threshold=0.85
print("임시 판별 기준:",threshold,"(검증되지 않음)")

## STEP 11. 테스트 사진 업로드 및 OWNER 후보 판별

In [ ]:
uploaded=files.upload()
assert uploaded
test_name=next(iter(uploaded))
test_image=cv2.imread(test_name)
assert test_image is not None
boxes=face_detector.detectMultiScale(cv2.cvtColor(test_image,cv2.COLOR_BGR2GRAY),scaleFactor=1.1,minNeighbors=5,minSize=(40,40))
assert len(boxes)>0
x,y,w,h=max(boxes,key=lambda b:b[2]*b[3])
test_face=cv2.resize(cv2.cvtColor(test_image[y:y+h,x:x+w],cv2.COLOR_BGR2RGB),(224,224))
plt.imshow(test_face);plt.axis("off");plt.show()
test_X=preprocess_input(np.expand_dims(test_face.astype(np.float32),0))
test_feature=normalize(base_model.predict(test_X,verbose=0))[0]
similarity=float(np.dot(test_feature,owner_vector))
print("유사도:",round(similarity,4),"판별:","OWNER 후보" if similarity>=threshold else "UNKNOWN 후보")

## STEP 12. 모델 저장·다운로드·재검증

In [ ]:
base_model.save("mobilenet_face.keras")
np.savez("owner_face.npz",owner_vector=owner_vector,threshold=threshold)
files.download("mobilenet_face.keras")
files.download("owner_face.npz")
loaded_model=tf.keras.models.load_model("mobilenet_face.keras")
saved=np.load("owner_face.npz")
new_feature=normalize(loaded_model.predict(test_X,verbose=0))[0]
new_score=float(np.dot(new_feature,saved["owner_vector"]))
print("로드된 모델:",loaded_model.output_shape,"점수:",round(new_score,4))
print("결과:","OWNER 후보" if new_score>=float(saved["threshold"]) else "UNKNOWN 후보")

## 주의사항

- 0.85는 임시 시연 기준이며 성능 검증 결과가 아닙니다.
- ImageNet MobileNetV2는 얼굴 인식 특화 모델이 아닙니다.
- 데이터 증강 40장은 독립된 40명의 얼굴이 아닙니다.
- 사용하지 않은 주인/타인 사진을 이용한 별도 평가가 필요합니다.
- 보안·출입통제에 사용하지 마세요.